# Base cohort

Make a Dask-based base cohort with essential sociodemographic fields

In [ ]:
!python -V

In [ ]:

import os
import sys

root_path = os.path.dirname(os.path.abspath(os.path.dirname('__file__')))
sys.path.insert(0, root_path)

In [ ]:

from util.general_utils import field_availability_check, list_field_instances
from util.dask_utils import import_field_from_dask
from util.parquet_maker import dask_to_parquet
from env.parameters import P
from util.logging import logging_setup

In [ ]:
import importlib

In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_rows', 500)

In [ ]:
import dask.dataframe as dd

# Patient IDs

In [ ]:
print(P.output_csv_path)
P.eid_included_name

In [ ]:
df_eid_included = pd.read_csv(f'''{P.output_csv_path}/{P.eid_included_name}.csv''')
df_eid_included.head()

In [ ]:
df_eid_included.count()

# # Load the core data and check the field

In [ ]:
dd_all = dd.read_parquet(f'''{P.output_parquet_path}/{P.core_all_name}''')
dd_all.head()

In [ ]:
cols_all = dd_all.columns

In [ ]:
cols_all

In [ ]:
#data of birth
search_term = "33"
#field_availability_check(col_list=cols_all, search_for_field=search_term)
list_term = list_field_instances(col_list=cols_all, search_for_field=search_term)
list_term

# DOB

In [ ]:
#month of birth
search_term = "52"
list_term = list_field_instances(col_list=cols_all, search_for_field=search_term)
print(list_term)
dd_mob = import_field_from_dask(dd_in= dd_all, field_name=search_term, eid_col="eid")

# to pandas
df_mob = dd_mob.compute()
print(df_mob.dtypes)


In [ ]:
# typecase
# eid and mob are integer
df_mob = df_mob.astype('int64')
print(df_mob.dtypes)
df_mob.head()

In [ ]:
#year of birth
search_term = "34"
list_term = list_field_instances(col_list=cols_all, search_for_field=search_term)
print(list_term)
dd_yob = import_field_from_dask(dd_in= dd_all, field_name=search_term, eid_col="eid")
# to pandas
df_yob = dd_yob.compute()
print(df_yob.dtypes)


In [ ]:
# typecase

# eid and yob are integer
df_yob = df_yob.astype('int64')
print(df_yob.dtypes)
df_yob.head()

In [ ]:
#year of birth
search_term = "22200"
list_term = list_field_instances(col_list=cols_all, search_for_field=search_term)
print(list_term)
dd_yob2 = import_field_from_dask(dd_in= dd_all, field_name=search_term, eid_col="eid")
# to pandas
df_yob2 = dd_yob2.compute()
print(df_yob2.dtypes)

In [ ]:
df_yob2.head()

In [ ]:
df_yob2[df_yob2['22200-0.0'].isna()]

In [ ]:
# typecase
# eid and yob are integer
# yob2 has nulls. Int64 is nullable.
df_yob2['eid'] = df_yob2['eid'].astype('int64')
df_yob2['22200-0.0'] = pd.to_numeric(df_yob2['22200-0.0'], errors='coerce').astype('Int64')

In [ ]:
df_yob2.dtypes

In [ ]:
df_dob = pd.merge(df_yob, df_yob2, on= 'eid', how="outer")
df_dob = pd.merge(df_dob, df_mob, on='eid', how="outer")
df_dob.head()

In [ ]:
# month is not null
df_dob[df_dob['52-0.0'].isnull()].shape

In [ ]:
df_dob[df_dob['34-0.0'].isnull()].shape

In [ ]:
# Just use 34 and 52
df_dob.dtypes

In [ ]:
df_dob['dob'] = pd.to_datetime(df_dob['34-0.0'].astype(str) + '-' +
                                df_dob['52-0.0'].astype(str) + '-' +
                                "01")
df_dob.head()

# Make the cohort (withdrawals applied)

In [ ]:
print(df_eid_included.shape[0])

In [ ]:
df_cohort = pd.merge(df_eid_included, df_dob[['eid', 'dob']], on="eid", how="left")

In [ ]:

print(df_cohort.shape)

# DOD

In [ ]:
# date of death
search_term = "40000"
list_term = list_field_instances(col_list=cols_all, search_for_field=search_term)
print(list_term)
dd_death_sc = import_field_from_dask(dd_in= dd_all, field_name=search_term, eid_col="eid")
# to pandas
df_death_sc = dd_death_sc.compute()
print(df_death_sc.dtypes)
print(df_death_sc.head())


In [ ]:
df_death_sc = df_death_sc[(~df_death_sc['40000-0.0'].isnull()) | 
(~df_death_sc['40000-1.0'].isnull())]
df_death_sc.head()

In [ ]:
list_term

In [ ]:
# typecase
# eid is non-nullable integer
# both ['40000-0.0', '40000-1.0'] are of date type
df_death_sc['eid'] = df_death_sc['eid'].astype('int64')
df_death_sc[list_term] = df_death_sc[list_term].apply(pd.to_datetime)
print(df_death_sc.dtypes)

In [ ]:
print(df_death_sc.shape)
print(df_death_sc[df_death_sc['40000-0.0'].isnull()].shape)
print(df_death_sc[df_death_sc['40000-1.0'].isnull()].shape)


In [ ]:
df_death_sc[~df_death_sc['40000-1.0'].isnull()].head()

In [ ]:
df_death_sc[(~df_death_sc['40000-1.0'].isnull()) & (df_death_sc['40000-1.0'] != df_death_sc['40000-0.0'])].head()

In [ ]:
# Strategy: use 40000-0.0

In [ ]:
P.input_file_dict.get("death")

In [ ]:
# check death data
#path_deaths = "~/Documents/Biobank_2023/data_portal/death.txt"
path_deaths = P.input_file_dict.get("death").get("path")

df_death = pd.read_csv(path_deaths, delimiter='\t')
df_death.head()

In [ ]:
df_death.dtypes

In [ ]:
df_death['dod'] = pd.to_datetime(df_death['date_of_death'], format="%d/%m/%Y")
df_death.head()

In [ ]:
df_death_sc['40000-0.0'] = pd.to_datetime(df_death_sc['40000-0.0'])
df_death_sc.head()

In [ ]:

df_death_mix = pd.merge(df_death[["eid", "dod"]], df_death_sc[['eid', '40000-0.0']], on= "eid", how="outer")
df_death_mix.head()

In [ ]:
df_death_mix.dtypes

In [ ]:
df_death_mix[df_death_mix['dod'].isnull()].shape

In [ ]:
df_death_mix[(df_death_mix['dod'].isnull()) & (~df_death_mix['40000-0.0'].isnull())].shape

In [ ]:
df_death_mix.loc[(df_death_mix['dod'].isnull()) & (~df_death_mix['40000-0.0'].isnull()), "dod"] = df_death_mix['40000-0.0']

In [ ]:
df_death_mix[df_death_mix['dod'].isnull()].shape


In [ ]:
df_death_mix.head()

In [ ]:
# check uniqueness
print(df_death_mix.shape)
print(df_death_mix['eid'].nunique())

In [ ]:
# Make unique
df_death_clean = df_death_mix.copy()
df_death_clean['eid_group'] = df_death_clean.groupby('eid')['eid'].transform('count')
df_death_clean[df_death_clean['eid_group']==2]

In [ ]:
df_death_clean['eid_rank'] = df_death_clean.groupby('eid').cumcount()+1
df_death_clean[df_death_clean['eid_group']==2].head()


In [ ]:
# Keep the first one
df_death_clean = df_death_clean[df_death_clean["eid_rank"]==1]
print(df_death_clean.shape)
print(df_death_clean['eid'].nunique())

In [ ]:
print(df_cohort.shape)
print(df_cohort['eid'].nunique())

In [ ]:
#merge
df_cohort = pd.merge(df_cohort, df_death_clean[["eid", "dod"]], on="eid", how="left")
print(df_cohort.shape)
print(df_cohort['eid'].nunique())

In [ ]:
# Death flag
df_cohort["isin_deaths"] = np.where(df_cohort['dod'].isna(), 0, 1)

In [ ]:
df_cohort.head(20)

In [ ]:
df_cohort.groupby("isin_deaths").size()

# Add Sex

In [ ]:
P.field_encoding_dict # coding of sex


In [ ]:
sex_coding = pd.read_csv(P.field_encoding_dict .get("sex").get("coding_file_path"), delimiter="\t")
sex_coding.head(10)

In [ ]:
#sex
search_term = "31"
list_term = list_field_instances(col_list=cols_all, search_for_field=search_term)
print(list_term)
dd_sex = import_field_from_dask(dd_in= dd_all, field_name=search_term, eid_col="eid")
# to pandas
df_sex = dd_sex.compute()
print(df_sex.dtypes)
print(df_sex.head())

In [ ]:
# typecase
# eid and sex arenon-nullable integer
# both ['40000-0.0', '40000-1.0'] are of date type
df_sex = df_sex.astype('int64')
print(df_sex.dtypes)


In [ ]:
df_sex.groupby("31-0.0").size()


In [ ]:
df_sex['31-0.0'].value_counts(dropna=False)

In [ ]:
# map
sex_map_dict = sex_coding.set_index("coding").to_dict()['meaning']
print(sex_map_dict)


In [ ]:
df_sex.loc[:, 'sex'] = df_sex['31-0.0'].map(sex_map_dict)
df_sex.head()

In [ ]:
df_sex['sex_code'] = df_sex['31-0.0']

In [ ]:
df_cohort = pd.merge(df_cohort, df_sex[["eid", "sex", "sex_code"]], on = "eid", how="left")
print(df_cohort.shape)
print(df_cohort['eid'].nunique())


In [ ]:
df_cohort.head()

# # quality control

In [ ]:
# unique id?
print(df_cohort.shape)
print(df_cohort['eid'].nunique())

In [ ]:
# any null dob?
df_cohort['dob'].isnull().sum()

In [ ]:
# any null sex?
df_cohort['sex'].isnull().sum()


# Add data and age start,  and end date

In [ ]:
# load the date of instances

In [ ]:
#Date of attending assessment centre
search_term = "53"
list_term = list_field_instances(col_list=cols_all, search_for_field=search_term)
print(list_term)
dd_date_assess = import_field_from_dask(dd_in= dd_all, field_name=search_term, eid_col="eid")
# to pandas
df_date_assess = dd_date_assess.compute()
print(df_date_assess.dtypes)
print(df_date_assess.head())

In [ ]:
df_date_assess.shape[0]

In [ ]:
print(df_date_assess[df_date_assess["53-0.0"].isnull()].shape[0])
print(df_date_assess[df_date_assess["53-1.0"].isnull()].shape[0])
print(df_date_assess[df_date_assess["53-2.0"].isnull()].shape[0])
print(df_date_assess[df_date_assess["53-3.0"].isnull()].shape[0])


In [ ]:
df_date_assess.dtypes

In [ ]:
df_date_assess['evdt_ukb_start'] = pd.to_datetime(df_date_assess["53-0.0"])
df_date_assess['eid'] = df_date_assess["eid"].astype("int64")


In [ ]:
df_date_assess.dtypes

In [ ]:
np.min(df_date_assess['evdt_ukb_start'])

In [ ]:
df_cohort = pd.merge(df_cohort, df_date_assess[["eid", "evdt_ukb_start"]], on = "eid", how="left")

In [ ]:
df_cohort[df_cohort['dob']> df_cohort['evdt_ukb_start']].shape


In [ ]:
df_cohort[df_cohort['evdt_ukb_start'].isnull()].shape[0]

In [ ]:
df_cohort["age_ukb_start"] = ((df_cohort["evdt_ukb_start"]  - df_cohort["dob"])/ pd.Timedelta(days=365.25)).round().astype("Int64")

In [ ]:
df_cohort.head()


In [ ]:
df_cohort["age_ukb_start"].hist()

In [ ]:
P.data_end_date

In [ ]:
df_cohort['evdt_ukb_end'] = P.data_end_date
df_cohort['evdt_ukb_end'] = pd.to_datetime(df_cohort["evdt_ukb_end"])
df_cohort

In [ ]:
df_cohort[df_cohort['dob']> df_cohort['evdt_ukb_end']].shape

In [ ]:
df_cohort[(df_cohort['dod'].notnull()) & (df_cohort['dod']> df_cohort['evdt_ukb_end'])].shape


In [ ]:
df_cohort.to_csv(f'''{P.output_csv_path}/cohort_1_base.csv''', index=False)


In [ ]:
df = pd.read_csv(f'''{P.output_csv_path}/cohort_1_base.csv''')
df.head()

In [ ]:
df_cohort.head()

In [ ]:
# save as parquet
dd_cohort = dd.from_pandas(df_cohort, npartitions=2)

dask_to_parquet(dd_cohort, output_path=P.output_parquet_path, output_parquet_name="cohort_1_base", write_index=False)
